# SWAN Nested Grid: Model Setup & Execution

---

## Overview

This notebook configures and runs a **two-stage nested SWAN simulation** for the California Bay case study. The nested approach provides high-resolution wave fields in the WEC deployment zone while keeping computational cost manageable.

### Workflow

```
Stage 1: Coarse (Parent) Run
  └─ Full California Bay domain (120×216, ~460 m)
  └─ Forced by ERA5 wind + TPAR wave boundary
  └─ NESTOUT: saves spectral boundary along fine grid edges
  └─ Output: nest_boundary.bnd + coarse_block.mat

Stage 2: Fine (Nested) Run
  └─ High-energy deployment zone (300×300, ~100 m)
  └─ NESTINP: reads spectral boundary from Stage 1
  └─ Same wind forcing and physics as coarse grid
  └─ Output: fine_block.mat (hourly, high-resolution)
```

### Key SWAN Commands for Nesting

| Command | Stage | Purpose |
|---------|-------|--------|
| `NGRID` | Both | Define the nested grid boundary geometry |
| `NESTOUT` | Coarse | Write full spectral data along the nested grid boundary |
| `NESTINP` | Fine | Read spectral boundary conditions from the coarse run |

### Important Notes

- **One-way nesting**: Information flows from coarse → fine only. The fine grid does not feed back.
- **`NGRID` must match**: The `NGRID` definition must be identical in both input files (same origin, extent, and resolution).
- **Physics consistency**: Both grids should use the same physics settings to avoid discontinuities at the nesting boundary.
- **Temporal resolution**: The `NESTOUT` interval should match or exceed the fine grid's needs (we use 1 HR).

### Prerequisites

- SWAN must be installed and available on PATH
- `swan_runner.py` module (in `code/`)
- California Bay data files (GEBCO, wind, boundary conditions)

### Dependencies

```
numpy, scipy, matplotlib
```

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Import swan_runner
CODE_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'code')
sys.path.insert(0, CODE_DIR)
import swan_runner

# Model directory for the nested grid
MODEL_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'california_bay_nested.F')

SWAN_CMD = 'swanrun'
SWAN_NPROC = 6

print(f"SWAN available: {swan_runner.check_swan_available(SWAN_CMD)}")
print(f"Model directory: {os.path.basename(MODEL_DIR)}")
print(f"\nFiles in model directory:")
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, f))
    link = ' -> ' + os.readlink(os.path.join(MODEL_DIR, f)) if os.path.islink(os.path.join(MODEL_DIR, f)) else ''
    print(f"  {f:30s} {size:>12,} bytes{link}")

## 1. Review SWAN Input Files

### 1.1 Coarse Grid Input (`INPUT_coarse`)

The coarse input is identical to the original California Bay model, with two additions:
1. **`NGRID` + `NESTOUT`**: Defines the nested grid boundary and writes spectral data along it
2. **`BLOCK` output**: Saves the 2-D wave field for comparison with the fine grid

The `NGRID` command specifies the geometry of the nested boundary:
```
NGRID 'nest_zone' xpc ypc alpc xlenc ylenc mxn myn
```
where `mxn` and `myn` define how many points SWAN uses to discretize the boundary (not the fine grid's computational cells).

In [ ]:
coarse_input = os.path.join(MODEL_DIR, 'INPUT_coarse')
with open(coarse_input) as f:
    content = f.read()
print("=" * 65)
print("INPUT_coarse")
print("=" * 65)
print(content)

### 1.2 Fine Grid Input (`INPUT_fine`)

The fine grid input has a different `CGRID` (higher resolution, smaller domain) and replaces the western boundary condition (`BOUndspec SIDE W`) with `NESTINP`, which reads the spectral boundary from the coarse run.

Key differences from the coarse input:
- `CGRID`: 300×300 cells at ~100 m resolution, no rotation
- `NESTINP` instead of `BOUndspec`: Boundary comes from the coarse run, not ERA5
- `BLOCK` output at **hourly** intervals (vs 6-hourly for coarse) to capture detailed temporal variability

In [ ]:
fine_input = os.path.join(MODEL_DIR, 'INPUT_fine')
with open(fine_input) as f:
    content = f.read()
print("=" * 65)
print("INPUT_fine")
print("=" * 65)
print(content)

## 2. Stage 1: Run the Coarse (Parent) Grid

The coarse run solves the spectral action balance equation over the full California Bay domain. At each output timestep, SWAN writes the 2-D directional wave spectrum along the fine grid boundary to `nest_boundary.bnd`.

This file contains the full spectral information (energy density as a function of frequency and direction) — not just bulk parameters like $H_s$ and $T_p$. This is critical because the fine grid needs the complete spectral shape to properly account for swell and wind-sea components arriving from different directions.

**Expected runtime**: Several minutes depending on your system (the grid has 25,920 cells × 1440 timesteps).

In [ ]:
if not swan_runner.check_swan_available(SWAN_CMD):
    print(f"WARNING: '{SWAN_CMD}' not found on PATH.")
    print("Install SWAN and re-run. See the repository README.")
else:
    print("=" * 65)
    print("Stage 1: Running COARSE (parent) grid...")
    print("=" * 65)
    
    result = swan_runner.run_swan(
        MODEL_DIR, input_name='INPUT_coarse',
        swan_cmd=SWAN_CMD, nproc=SWAN_NPROC
    )
    
    # Check that the nest boundary file was created
    nest_file = os.path.join(MODEL_DIR, 'nest_boundary.bnd')
    if os.path.exists(nest_file):
        size_mb = os.path.getsize(nest_file) / 1e6
        print(f"\n  nest_boundary.bnd created: {size_mb:.1f} MB")
    else:
        print("\n  WARNING: nest_boundary.bnd not found!")
    
    coarse_block = os.path.join(MODEL_DIR, 'coarse_block.mat')
    if os.path.exists(coarse_block):
        size_mb = os.path.getsize(coarse_block) / 1e6
        print(f"  coarse_block.mat created: {size_mb:.1f} MB")
    
    print("\n  Stage 1 complete.")

## 3. Stage 2: Run the Fine (Nested) Grid

The fine grid reads the spectral boundary from `nest_boundary.bnd` and runs at ~100 m resolution over the deployment zone. The boundary conditions are applied along all four edges of the fine grid, providing spatially and temporally varying wave input from the coarse solution.

Because the fine grid has many more cells per unit area ($300 \times 300 = 90{,}000$ cells), it may take longer than the coarse run despite covering a smaller domain. The hourly output interval also increases the output file size.

**Expected runtime**: Several minutes (90,000 cells × 1440 timesteps).

In [ ]:
nest_file = os.path.join(MODEL_DIR, 'nest_boundary.bnd')

if not swan_runner.check_swan_available(SWAN_CMD):
    print(f"WARNING: '{SWAN_CMD}' not found on PATH.")
elif not os.path.exists(nest_file):
    print("WARNING: nest_boundary.bnd not found.")
    print("Run Stage 1 (coarse grid) first.")
else:
    print("=" * 65)
    print("Stage 2: Running FINE (nested) grid...")
    print("=" * 65)
    
    result = swan_runner.run_swan(
        MODEL_DIR, input_name='INPUT_fine',
        swan_cmd=SWAN_CMD, nproc=SWAN_NPROC
    )
    
    fine_block = os.path.join(MODEL_DIR, 'fine_block.mat')
    if os.path.exists(fine_block):
        size_mb = os.path.getsize(fine_block) / 1e6
        print(f"\n  fine_block.mat created: {size_mb:.1f} MB")
    
    print("\n  Stage 2 complete.")

## 4. Compare Results: Coarse vs Fine Grid

After both runs complete, we compare the wave fields from the coarse and fine grids over the nested domain. This reveals the additional detail that the fine grid provides:
- Sharper refraction patterns
- Better-resolved wave shadow zones
- More precise energy hotspot locations

We read both `.MAT` block outputs and overlay them.

In [ ]:
import warnings

coarse_block_file = os.path.join(MODEL_DIR, 'coarse_block.mat')
fine_block_file = os.path.join(MODEL_DIR, 'fine_block.mat')

has_coarse = os.path.exists(coarse_block_file)
has_fine = os.path.exists(fine_block_file)

print(f"Coarse block output: {'OK' if has_coarse else 'MISSING (run Stage 1)'}")
print(f"Fine block output:   {'OK' if has_fine else 'MISSING (run Stage 2)'}")

if has_coarse and has_fine:
    # Read coarse output
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mat_c = swan_runner.read_mat_block(coarse_block_file)
    Xp_c, Yp_c = mat_c['Xp'], mat_c['Yp']
    Hs_c = swan_runner.extract_block_variable(mat_c, 'Hsig')
    Hs_c = np.where(Hs_c < 0, np.nan, Hs_c)
    Hs_mean_c = np.nanmean(Hs_c, axis=0)
    print(f"\nCoarse grid: {Xp_c.shape[1]}x{Xp_c.shape[0]}, "
          f"{Hs_c.shape[0]} timesteps")
    
    # Read fine output
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mat_f = swan_runner.read_mat_block(fine_block_file)
    Xp_f, Yp_f = mat_f['Xp'], mat_f['Yp']
    Hs_f = swan_runner.extract_block_variable(mat_f, 'Hsig')
    Hs_f = np.where(Hs_f < 0, np.nan, Hs_f)
    Hs_mean_f = np.nanmean(Hs_f, axis=0)
    print(f"Fine grid:   {Xp_f.shape[1]}x{Xp_f.shape[0]}, "
          f"{Hs_f.shape[0]} timesteps")
    
    # --- Side-by-side comparison over the fine grid domain ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # Common color scale
    vmin = 0
    vmax = max(np.nanmax(Hs_mean_f), np.nanpercentile(Hs_mean_c, 99))
    levels = np.linspace(vmin, vmax, 25)
    
    # Fine grid domain limits for zooming the coarse plot
    fine_lon = (Xp_f.min(), Xp_f.max())
    fine_lat = (Yp_f.min(), Yp_f.max())
    
    # Coarse grid (zoomed to fine domain)
    land_c = np.isnan(Hs_mean_c)
    Hs_plot_c = np.ma.masked_where(land_c, Hs_mean_c)
    cf1 = ax1.contourf(Xp_c, Yp_c, Hs_plot_c, levels=levels,
                       cmap='YlOrRd', extend='both')
    ax1.contourf(Xp_c, Yp_c, np.where(land_c, 1, np.nan),
                 levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
    ax1.contour(Xp_c, Yp_c, np.where(land_c, 1, 0),
                levels=[0.5], colors='k', linewidths=0.8, zorder=3)
    ax1.set_xlim(fine_lon)
    ax1.set_ylim(fine_lat)
    ax1.set_title('Coarse Grid (~460 m)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Longitude (\u00b0E)')
    ax1.set_ylabel('Latitude (\u00b0N)')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.15, ls='--')
    plt.colorbar(cf1, ax=ax1, shrink=0.8, label='Mean $H_s$ (m)')
    
    # Fine grid
    land_f = np.isnan(Hs_mean_f)
    Hs_plot_f = np.ma.masked_where(land_f, Hs_mean_f)
    cf2 = ax2.contourf(Xp_f, Yp_f, Hs_plot_f, levels=levels,
                       cmap='YlOrRd', extend='both')
    ax2.contourf(Xp_f, Yp_f, np.where(land_f, 1, np.nan),
                 levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
    ax2.contour(Xp_f, Yp_f, np.where(land_f, 1, 0),
                levels=[0.5], colors='k', linewidths=0.8, zorder=3)
    ax2.set_title('Fine Nested Grid (~100 m)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Longitude (\u00b0E)')
    ax2.set_ylabel('Latitude (\u00b0N)')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.15, ls='--')
    plt.colorbar(cf2, ax=ax2, shrink=0.8, label='Mean $H_s$ (m)')
    
    fig.suptitle('Nested Grid Comparison \u2014 Mean $H_s$ over Deployment Zone',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    
    # Statistics comparison
    print(f"\nStatistics over fine grid domain:")
    print(f"  Coarse: mean Hs = {np.nanmean(Hs_mean_c):.3f} m, "
          f"max = {np.nanmax(Hs_mean_c):.3f} m")
    print(f"  Fine:   mean Hs = {np.nanmean(Hs_mean_f):.3f} m, "
          f"max = {np.nanmax(Hs_mean_f):.3f} m")
else:
    print("\nRun both stages to compare results.")

## 5. Wave Power Map on Fine Grid

The fine grid provides the high-resolution wave power density field that is most useful for WEC array optimization. This is the field that the GA optimizer (notebook `04_optimization/`) would use for the coupled optimization.

In [ ]:
if has_fine:
    Tp_f = swan_runner.extract_block_variable(mat_f, 'RTpeak')
    Tp_f = np.where(Tp_f < 0, np.nan, Tp_f)
    Tp_mean_f = np.nanmean(Tp_f, axis=0)
    
    # Wave power density (kW/m)
    rho_g2 = 1025 * 9.81**2 / (64 * np.pi)
    P_fine = rho_g2 * Hs_mean_f**2 * Tp_mean_f / 1000
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    P_plot = np.ma.masked_where(land_f, P_fine)
    vmax_p = np.nanpercentile(P_fine[~land_f], 99)
    levels_p = np.linspace(0, vmax_p, 25)
    cf = ax.contourf(Xp_f, Yp_f, P_plot, levels=levels_p,
                     cmap='inferno', extend='max')
    plt.colorbar(cf, ax=ax, shrink=0.8, label='Mean Wave Power Density (kW/m)')
    
    ax.contourf(Xp_f, Yp_f, np.where(land_f, 1, np.nan),
                levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
    ax.contour(Xp_f, Yp_f, np.where(land_f, 1, 0),
               levels=[0.5], colors='k', linewidths=0.8, zorder=3)
    
    ax.set_title('Wave Power Density \u2014 Fine Nested Grid (~100 m resolution)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude (\u00b0E)', fontsize=11)
    ax.set_ylabel('Latitude (\u00b0N)', fontsize=11)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.15, ls='--')
    plt.tight_layout()
    plt.show()
    
    print(f"Fine grid wave power:")
    print(f"  Max:  {np.nanmax(P_fine):.1f} kW/m")
    print(f"  Mean: {np.nanmean(P_fine[~land_f]):.1f} kW/m")
else:
    print("Fine grid output not available. Run Stage 2 first.")

## Summary

This notebook demonstrated the complete SWAN nested grid workflow:

| Stage | Grid | Resolution | Key Output |
|-------|------|-----------|------------|
| 1 (Coarse) | 120×216 | ~460 m | `nest_boundary.bnd` (spectral BC) + `coarse_block.mat` |
| 2 (Fine) | 300×300 | ~100 m | `fine_block.mat` (high-res wave field) |

### SWAN Nesting Commands Reference

```
! In the COARSE input file:
NGRID 'name' xpc ypc alpc xlenc ylenc mxn myn
NESTOUT 'name' 'filename.bnd' OUTPUT tbegin dt_unit

! In the FINE input file:
NGRID 'name' xpc ypc alpc xlenc ylenc mxn myn   ! must match coarse
NESTINP 'name' 'filename.bnd' FREE
```

### When to Use Nesting

- **WEC site assessment**: High resolution around deployment sites for accurate power estimates
- **Coastal engineering**: Detailed wave conditions at harbor entrances, breakwaters
- **Sediment transport**: Fine-scale wave-driven currents in nearshore zones
- **Storm impact studies**: Detailed overtopping and inundation at specific locations

### Limitations

- **One-way only**: Processes in the fine grid (e.g., WEC wave absorption) don't affect the coarse grid
- **Bathymetry resolution**: GEBCO (~460 m) limits the fine grid's effective resolution; for true 100 m physics, higher-resolution bathymetry is needed
- **Computational cost**: The fine grid has ~3.5× more cells than the coarse grid, plus hourly output
- **Spectral boundary file size**: `nest_boundary.bnd` can be large (hundreds of MB to GB) for long simulations with many spectral bins